# An-Ra V4 — hardened TPU launch (`core-vnext`)

This notebook intentionally has one production path: clone the exact `core-vnext` branch and run `training.kaggle_core_vnext`.

Safety invariants:
- exact parent + pack verification before TPU workers spawn;
- canonical XLA latest checkpoint every 200 pack steps;
- sparse candidate saving disabled because the old candidate helper used plain `torch.save` on XLA-resident optimizer state;
- final checkpoint is strictly reloaded and compared with the parent;
- local SHA-256 export is verified;
- verified checkpoint is uploaded to a Kaggle Dataset and its remote manifest is downloaded back and checked before success is reported.


In [ ]:
# 1. Runtime preflight — fail closed.
import importlib.metadata, importlib.util, os
os.environ['PJRT_DEVICE'] = 'TPU'
if importlib.util.find_spec('torch_xla') is None:
    raise RuntimeError('TPU not attached. Select TPU v5e-8, restart, then Run All.')
import torch
print({'pjrt': os.environ['PJRT_DEVICE'], 'torch': torch.__version__,
       'torch_xla': importlib.metadata.version('torch-xla')})

In [ ]:
# 2. Clone/fetch the exact branch, bind provenance, and run the hardened launcher.
import json, os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_REF = 'core-vnext'
REPO = Path('/kaggle/working/anra')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', f'origin/{REPO_REF}'], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
os.environ['ANRA_SOURCE_COMMIT'] = SOURCE_COMMIT
print(json.dumps({'branch': REPO_REF, 'commit': SOURCE_COMMIT, 'repo': str(REPO)}, indent=2))
result = subprocess.run([sys.executable, '-m', 'training.kaggle_core_vnext'], cwd=REPO)
if result.returncode != 0:
    raise RuntimeError(f'hardened TPU launch exited {result.returncode}; inspect the logs above')